In [6]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate PyMuPDF python-docx

import os
import faiss
import numpy as np
import fitz
from docx import Document
from google.colab import files

print("Upload your Sanskrit files (.txt / .pdf / .docx)")
uploaded = files.upload()

# =========================
# LOADER
# =========================
def load_documents(uploaded_files):
    texts = []

    for filename in uploaded_files:
        try:
            content = ""

            if filename.endswith(".txt"):
                with open(filename, "r", encoding="utf-8", errors="ignore") as f:
                    content = f.read()

            elif filename.endswith(".pdf"):
                doc = fitz.open(filename)
                for page in doc:
                    content += page.get_text()

            elif filename.endswith(".docx"):
                doc = Document(filename)
                for para in doc.paragraphs:
                    content += para.text + "\n"

            else:
                print(f"❌ Unsupported file skipped: {filename}")
                continue

            if content.strip():
                texts.append(content)
                print(f"✅ Loaded: {filename}")
            else:
                print(f"⚠ Empty content: {filename}")

        except Exception as e:
            print(f"❌ Error reading {filename}: {e}")

    return texts

documents = load_documents(uploaded)

print(f"\nTotal documents: {len(documents)}")

if len(documents) == 0:
    raise ValueError("No valid documents loaded — CHECK FILE FORMAT")

Upload your Sanskrit files (.txt / .pdf / .docx)


Saving Rag-docs.docx to Rag-docs (1).docx
✅ Loaded: Rag-docs (1).docx

Total documents: 1


In [7]:
from sentence_transformers import SentenceTransformer

# =========================
# SANSKRIT CHUNKER
# =========================
def sanskrit_chunker(text, chunk_size=300):
    text = text.replace("॥", "।")
    sentences = [s.strip() for s in text.split("।") if s.strip()]

    chunks = []
    current = ""

    for sent in sentences:
        if len(current) + len(sent) < chunk_size:
            current += sent + "।"
        else:
            chunks.append(current.strip())
            current = sent + "।"

    if current:
        chunks.append(current.strip())

    return chunks

# =========================
# CREATE CHUNKS
# =========================
all_chunks = []
for doc in documents:
    all_chunks.extend(sanskrit_chunker(doc))

print("Total chunks:", len(all_chunks))

if len(all_chunks) == 0:
    raise ValueError("Chunking failed — document may be empty")

print("\nSample chunk:\n", all_chunks[0][:200])

# =========================
# EMBEDDING MODEL
# =========================
embedder = SentenceTransformer("intfloat/multilingual-e5-large")

# =========================
# EMBEDDINGS
# =========================
embeddings = embedder.encode(
    ["passage: " + t for t in all_chunks],
    convert_to_numpy=True,
    normalize_embeddings=True
)

embeddings = np.array(embeddings)

if len(embeddings.shape) == 1:
    embeddings = embeddings.reshape(1, -1)

print("Embedding shape:", embeddings.shape)

# =========================
# FAISS
# =========================
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("✅ FAISS ready")

Total chunks: 33

Sample chunk:
 मूर्खभृत्यस्य

"अरे शंखनाद, गच्छापणम्, शर्कराम् आनय।" इति स्वभृत्यम् शंखनादम् गोवर्धनदासः आदिशति।ततः शंखनादः आपणम् गच्छति, शर्कराम् जीर्णे वस्त्रे न्यस्यति च।तस्मात् जीर्णवस्त्रात् मार्गे एव सर्वापि श


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding shape: (33, 1024)
✅ FAISS ready


In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/mt5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("✅ mT5 loaded (fixed mode)")

# =========================
# RETRIEVE
# =========================
def retrieve(query, k=3):
    q_emb = embedder.encode(
        ["query: " + query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(q_emb, k)
    return [all_chunks[i] for i in indices[0]]


def build_prompt(query, context):
    return f"""
Extract the exact answer from the Sanskrit context.

Return ONLY a relevant sentence from the context.

Context:
{context}

Question:
{query}

Answer in Sanskrit:
"""


def extract_from_chunks(query, chunks):
    # simple keyword fallback if model fails
    for chunk in chunks:
        if any(word in chunk for word in query.split()):
            return chunk[:200]
    return "उत्तर उपलब्ध नहीं है"

# =========================
# QA FUNCTION
# =========================
def ask(query):
    chunks = retrieve(query)

    print("\n--- RETRIEVED ---")
    for i, c in enumerate(chunks):
        print(f"\nChunk {i+1}:\n{c}")

    context = "\n".join(chunks)
    prompt = build_prompt(query, context)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    if "<extra_id_" in answer or len(answer) < 5:
        answer = extract_from_chunks(query, chunks)

    print("\n--- ANSWER ---")
    print(answer)

# =========================
# LOOP
# =========================
while True:
    q = input("\nAsk (or exit): ")
    if q.lower() == "exit":
        break
    ask(q)

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 116, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 95, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 71, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 47, in spawn_con

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ mT5 loaded (fixed mode)

Ask (or exit): शंखनादः श्वानशावकम् कथं आनयत् ?

--- RETRIEVED ---

Chunk 1:
आज्ञापालकः शंखनादः श्वानशावकम् सन्चिकायाम् क्षिपति, सन्चिकाम् वस्त्रेण आच्छादयति च।तेन शावकस्य श्वासः रुध्दः भवति।सः च श्वानशावकः पञ्चत्वम् गच्छति।तदा गोवर्धनदासः शंखनादम् अभिधावति क्रोधेनाक्रोशति च, " धिक् मुढ, श्वानादिकम् दोरकेण बद्ध्वा आनयन्ति इति अपि नावगच्छसि किम् ?” इति।

Chunk 2:
ततः गोवर्धनदासः कोपेन शंखनादम् वदति, "अरे मूढ, कुत्रास्ति शर्करा ? शर्करादिकम् एवम् जीर्णेन वस्त्रेण न एवानयन्ति कदापि।इतःपरम् किमपि वस्तुजातम् दृढायाम् सन्चिकायाम् निक्षिप्य आनय च " इति।अत्रान्तरे गोवर्धनदासस्य पुत्रः "श्वानशावकम् आनय" इति शंखनादम् आदिशति।

Chunk 3:
मूर्खभृत्यस्य

"अरे शंखनाद, गच्छापणम्, शर्कराम् आनय।" इति स्वभृत्यम् शंखनादम् गोवर्धनदासः आदिशति।ततः शंखनादः आपणम् गच्छति, शर्कराम् जीर्णे वस्त्रे न्यस्यति च।तस्मात् जीर्णवस्त्रात् मार्गे एव सर्वापि शर्करा स्त्रवति।

--- ANSWER ---
आज्ञापालकः शंखनादः श्वानशावकम् सन्चिकायाम् क्षिपति, सन्चिकाम् वस्त्रेण आच्छादयति च।तेन शावकस्य श्वासः रुध्दः 

In [9]:
def split_sentences(text):
    text = text.replace("॥", "।")
    sentences = [s.strip() + "।" for s in text.split("।") if s.strip()]
    return sentences


def smart_answer(query, chunks):
    query_emb = embedder.encode(
        ["query: " + query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    best_score = -1
    best_sentence = ""

    for chunk in chunks:
        sentences = split_sentences(chunk)

        sent_embs = embedder.encode(
            ["passage: " + s for s in sentences],
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        scores = np.dot(sent_embs, query_emb)

        max_idx = np.argmax(scores)

        if scores[max_idx] > best_score:
            best_score = scores[max_idx]
            best_sentence = sentences[max_idx]

    return best_sentence if best_sentence else "उत्तर उपलब्ध नहीं है"


def ask_improved(query):
    chunks = retrieve(query)

    print("\n--- RETRIEVED ---")
    for i, c in enumerate(chunks):
        print(f"\nChunk {i+1}:\n{c}")

    answer = smart_answer(query, chunks)

    print("\n--- IMPROVED ANSWER ---")
    print(answer)

while True:
    q = input("\nAsk improved (or exit): ")
    if q.lower() == "exit":
        break
    ask_improved(q)


Ask improved (or exit): भक्तः किमर्थं न रक्षितः ?

--- RETRIEVED ---

Chunk 1:
तदा भक्तः उक्तवान् "भवतः साहाय्यम् न आवश्यकम्।देवः अस्ति।सः एव साहाय्यम् करोति" इति।तदा सज्जनः गतवान्।किंचित समयानंतरम्, अन्यः एकः आगतवान्।सः अपि तदैव पृष्टवान् "भो मित्र, किंचित् साहाय्यम् आवश्यकम् वा ?" इति।तदा भक्तः उक्तवान् "भवतः साहाय्यम् न आवश्यकम्।देवः अस्ति।सः एव साहाय्यम् करोति" इति।

Chunk 2:
यदि भवान् प्रयत्नम् एव न करोति चेत्, अहम् कथम् साहाय्यम् करोमि ?”
तदा भक्तस्य ज्ञानोदयः अभवत्।यदि वयम् प्रयत्नम् कुर्मः, तर्हि एव देवः साहाय्यम् करोति।उद्यमः साहसम् धैर्यम् बुद्धिः शक्तिः पराक्रमः।षडेते यत्र वर्तन्ते तत्र देवः साहाय्यकृत्।शीतं बहु बाधति।

Chunk 3:
तदा सः स्वर्गम् गतवान्।सः देवम् पृष्टवान् "देव, अहम् भवतः परमभक्तः।यदा मम कष्टः अभवत्, भवान् किंचित् अपि साहाय्यम् न कृतवान्।भवान् द्रष्टुम् अपि न आगतवान्।किमर्थम् ?”
तदा देवः उक्तवान् "भो भक्त, अहम् त्रिवारम् आगतवान्।पृष्टवान् च।'साहाय्यम् अवश्यकम् वा ?” इति।परंतु भवान् न स्वीकृतवान्।

--- IMPROVED ANSWER ---
तदा भक्तः उक्तवान् "भवतः साहाय्यम् न आव

In [10]:
test_queries = [
    "श्वानशावकस्य किं अभवत् ?",
    "शंखनादः कुत्र गच्छति ?",
    "वानराः किं कुर्वन्ति ?"
]

print("\n===== DEMO OUTPUT =====\n")

for q in test_queries:
    print(f"\n🔹 Question: {q}")

    chunks = retrieve(q)

    # show top chunk only (clean demo)
    print("\nTop Context:")
    print(chunks[0][:200], "...")

    # use improved smart answer
    answer = smart_answer(q, chunks)

    print("\nAnswer:")
    print(answer)

    print("\n" + "="*50)


===== DEMO OUTPUT =====


🔹 Question: श्वानशावकस्य किं अभवत् ?

Top Context:
आज्ञापालकः शंखनादः श्वानशावकम् सन्चिकायाम् क्षिपति, सन्चिकाम् वस्त्रेण आच्छादयति च।तेन शावकस्य श्वासः रुध्दः भवति।सः च श्वानशावकः पञ्चत्वम् गच्छति।तदा गोवर्धनदासः शंखनादम् अभिधावति क्रोधेनाक्रोशति च,  ...

Answer:
तदा गोवर्धनदासः शंखनादम् अभिधावति क्रोधेनाक्रोशति च, " धिक् मुढ, श्वानादिकम् दोरकेण बद्ध्वा आनयन्ति इति अपि नावगच्छसि किम् ?” इति।


🔹 Question: शंखनादः कुत्र गच्छति ?

Top Context:
आज्ञापालकः शंखनादः श्वानशावकम् सन्चिकायाम् क्षिपति, सन्चिकाम् वस्त्रेण आच्छादयति च।तेन शावकस्य श्वासः रुध्दः भवति।सः च श्वानशावकः पञ्चत्वम् गच्छति।तदा गोवर्धनदासः शंखनादम् अभिधावति क्रोधेनाक्रोशति च,  ...

Answer:
ततः गोवर्धनदासः कोपेन शंखनादम् वदति, "अरे मूढ, कुत्रास्ति शर्करा ? शर्करादिकम् एवम् जीर्णेन वस्त्रेण न एवानयन्ति कदापि।


🔹 Question: वानराः किं कुर्वन्ति ?

Top Context:
तदा चिन्ताकुलः नृपः उदघोषयत् "यः घण्टकर्णं नाशयेत्, तस्मै विपुलं सुवर्णं यच्छेयम् अहम्" इति।तत् श्रुत्वा काचन वृद्धा वनं गता।तत्र च कंचित् का